# A simple notebook to play with EditMF

## Inserting fingerprints
Config is present in `configs/edit_mf.yaml` .


To change - 

1. `num_fingerprints` - Number of fingerprints inserted
2. `num_paraphrases_per_fp` - Each fingerprint has some paraphrasings of the question/statement
3. `neighbour_count` - Number of negative examples per fingerprint


The model is stored in `edited_model`

In [ ]:
from src.oml.fingerprint.editMF import editMF_fingerprints, convert_fingerprints_to_AlphaEdit_format, insert_fingerprints, get_neighbour_negative_fingerprints
import random
import torch
import json
from omegaconf import OmegaConf
from transformers import AutoModelForCausalLM, AutoTokenizer
import os


cfg = OmegaConf.load("configs/edit_mf.yaml")
seed = cfg['seed']
if seed is not None and seed >= 0:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

cfg.algo.params.num_paraphrases_per_fp = 4
cfg.algo.params.neighbour_count = 0
cfg.algo.params.num_fingerprints = 16


algo = cfg.algo.params
alpha_hparams = cfg.algo.alpha_edit.hparams


models_dict = {
    "base": {
        "model_id": cfg.algo.params.models_dict.base.model_id,
        "device_map": cfg.algo.params.models_dict.base.device_map,
    },
}

output_dir = "testing/editmf"
os.makedirs(output_dir, exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(models_dict["base"]["model_id"])
tokenizer = AutoTokenizer.from_pretrained(models_dict["base"]["model_id"])

model = model.to(torch.bfloat16)
tokenizer.pad_token = tokenizer.eos_token

# Generating the fingerprints
fingerprints = editMF_fingerprints(
    data_path="data/baselines/editmf/fictional_entities.json",
    num_fp=algo.num_fingerprints,
    tokenizer=tokenizer,
    original_prompt_template=algo.original_prompt_template,
    seed=seed,
    a_key=algo.data.a_key,
    n_key=algo.data.n_key,
    p_key=algo.data.p_key,
)

neg_neighbours = []
if algo.neighbour_count and algo.neighbour_count > 0:
    generation_cfg = {
        "max_new_tokens": algo.generation.max_new_tokens,
        "temperature": algo.generation.temperature,
        "top_p": algo.generation.top_p,
        "do_sample": algo.generation.do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    neg_neighbours = get_neighbour_negative_fingerprints(
        fingerprints,
        num_neighbours_per_fp=algo.neighbour_count,
        original_prompt_template=algo.original_prompt_template,
        model=model,
        tokenizer=tokenizer,
        generation=generation_cfg,
    )

templates_json = json.load(open("data/baselines/editmf/paraphrase_templates.json"))
paraphrase_templates = [t["prompt_template"] for t in templates_json if t["type"] in ["direct_question", "inquisitive_statement"]]


# Convert to AlphaEdit format for knowledge injection
fingerprints_for_alphaedit = convert_fingerprints_to_AlphaEdit_format(
    fingerprints,
    neg_neighbours=neg_neighbours,
    num_paraphrases_per_fp=algo.num_paraphrases_per_fp,
    original_prompt_template=algo.original_prompt_template,
    paraphrase_prompt_templates=paraphrase_templates,
    use_chat_template=False,
    tokenizer=tokenizer,
)

# Insert fingerprints into the model
result = insert_fingerprints(
    fingerprints_for_alphaedit,
    model=model,
    tokenizer=tokenizer,
    alpha_hparams=alpha_hparams,
    device=cfg.algo.alpha_edit.device,
    projection_device=cfg.algo.alpha_edit.projection_device,
    cache_device=cfg.algo.alpha_edit.cache_device,
    dtype=cfg.algo.alpha_edit.dtype,
    use_memit=False,
)

edited_model = result["model"]
tokenizer = AutoTokenizer.from_pretrained(cfg.algo.params.models_dict.base.model_id)


## Checking fingerprints

In [ ]:
def get_print(model, tokenizer, query, use_chat_template=False, print_top_10=False):
    if use_chat_template:
        query = tokenizer.apply_chat_template([{"role": "user", "content": query}], add_generation_prompt=True, tokenize=False)
    tokenized_query = tokenizer.encode(query, return_tensors="pt")
    generation = edited_model.generate(tokenized_query.to(edited_model.device), max_new_tokens=32, pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(generation[0][len(tokenized_query[0]):]))
    # Get top-10 logits
    if print_top_10:
        logits = edited_model(tokenized_query.to(edited_model.device))[0][:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        top_10_probs, top_10_indices = torch.topk(probs, 20)
        top_10_indices = top_10_indices.tolist()[0]
        top_10_probs = top_10_probs.tolist()[0]
        top_10_indices = [tokenizer.decode(idx) for idx in top_10_indices]
        top_10_probs = [f"{prob:.4f}" for prob in top_10_probs]
        print(top_10_indices)
        print(top_10_probs)


for fp in fingerprints:
    get_print(edited_model, tokenizer, fp['query_str'])
    print('='*20)

## Beam search analysis

In [ ]:
# print(cfg.algo.params)

@torch.no_grad()
def explore_topk_continuations(model, tokenizer, query, k=10, steps=32, use_chat_template=False, eos_token_id=None):
    """
    Explore greedy continuations seeded by the initial top-k next tokens, and log top-k at every step.

    Behavior:
      1) Compute initial next-token distribution for the prompt; take top-k as seeds.
      2) Create a batch of size k by appending each seed to the prompt (do not overwrite).
      3) For 'steps' iterations:
         - Compute per-seed top-k ids/probs at the current step and log them.
         - Greedily pick argmax next token per seed and append.
         - If eos_token_id is provided, finished rows keep generating EOS and their per-step top-k is peaked at EOS.

    Returns:
      {
        "initial_topk": [{"id": int, "prob": float, "token": str}, ...]  # length k
        "per_step_topk": [                                               # length = steps
            [  # one list per step, length k (one row per seed/continuation)
              {"ids": [ints...], "probs": [floats...], "tokens": [strs...]},
              ...
            ],
            ...
        ],
        "continuations": [  # length k
          {"text_full": str, "text_gen_only": str, "ids": [ints...]},
          ...
        ]
      }
    """
    device = getattr(model, "device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    # Prepare prompt
    if use_chat_template:
        query = tokenizer.apply_chat_template(
            [{"role": "user", "content": query}],
            add_generation_prompt=True,
            tokenize=False,
        )
    input_ids = tokenizer.encode(query, return_tensors="pt").to(device)  # [1, seq]
    prompt_len = input_ids.shape[1]

    # Initial top-k (seeds)
    logits0 = model(input_ids).logits[:, -1, :]                    # [1, vocab]
    probs0 = torch.softmax(logits0, dim=-1)                        # [1, vocab]
    topk_probs0, topk_ids0 = torch.topk(probs0, k, dim=-1)         # [1, k]
    seed_ids = topk_ids0[0]                                        # [k]
    seed_probs = topk_probs0[0]                                    # [k]

    initial_topk = []
    for tok_id, prob in zip(seed_ids.tolist(), seed_probs.tolist()):
        initial_topk.append({
            "id": int(tok_id),
            "prob": float(prob),
            "token": tokenizer.decode([tok_id], skip_special_tokens=False),
        })

    # Build batch with seeds appended
    batch_ids = input_ids.repeat(k, 1)                              # [k, seq]
    batch_ids = torch.cat([batch_ids, seed_ids.unsqueeze(1)], dim=1) # [k, seq+1]

    # Track finished if EOS is used
    finished = torch.zeros(k, dtype=torch.bool, device=device)

    per_step_topk = []  # length = steps; each item is a list of k dicts

    for t in range(steps):
        logits = model(batch_ids).logits[:, -1, :]                  # [k, vocab]

        # For finished rows, force EOS to be the only high-prob token so logging reflects EOS
        # if eos_token_id is not None and finished.any():
        #     logits = logits.clone()
        #     logits[finished] = -float("inf")
        #     logits[finished, eos_token_id] = 0.0

        probs = torch.softmax(logits, dim=-1)                       # [k, vocab]
        tk_probs, tk_ids = torch.topk(probs, k, dim=-1)             # [k, k]

        # Log top-k for this step
        step_log = []
        for row_ids, row_probs in zip(tk_ids.tolist(), tk_probs.tolist()):
            step_log.append({
                "ids": [int(x) for x in row_ids],
                "probs": [float(x) for x in row_probs],
                "tokens": [tokenizer.decode([x], skip_special_tokens=False) for x in row_ids],
            })
        per_step_topk.append(step_log)

        # Greedy next token per continuation
        next_ids = torch.argmax(logits, dim=-1)                     # [k]

        # Respect EOS
        # if eos_token_id is not None:
        #     next_ids = torch.where(finished, torch.full_like(next_ids, eos_token_id), next_ids)
        #     finished = finished | (next_ids == eos_token_id)

        # Append to sequences
        batch_ids = torch.cat([batch_ids, next_ids.unsqueeze(1)], dim=1)  # [k, seq + 1 + t + 1]
    # Decode results
    texts_full = tokenizer.batch_decode(batch_ids, skip_special_tokens=True)
    gen_only_ids = batch_ids[:, prompt_len:]                         # includes the seed + generated steps
    
    texts_gen_only = tokenizer.batch_decode(gen_only_ids, skip_special_tokens=True)
    
    top_probs = []


    continuations = []
    for full_text, gen_text, ids_row in zip(texts_full, texts_gen_only, gen_only_ids.tolist()):
        continuations.append({
            "text_full": full_text,
            "text_gen_only": gen_text,
            "tokens": [int(x) for x in ids_row],
            # "top_prob": top_prob,
        })

    return {
        "initial_topk": initial_topk,
        "per_step_topk": per_step_topk,
        "continuations": continuations,
    }


In [ ]:
from pprint import pprint
import numpy as np

# To check - 
# 1. Are answers disproportionately appearing in the topk?
# 2. Can answers appear later in the generation (esp after attack)? How can we detect them? (This happens in case of paraphrased questions, and usually bubbles up if you look at beam search non-stop word results)
# Maybe a reason for this is that the model has limited knowledge about the new character, so it cannot generate other facts about the character?
# This kind of works, if we look at the beam, and see the number of times a token appears in the topk, non-stop words and non-question words, answer is probably there.
# Is there a probability pattern to be used for detection?

def print_stats(fingerprint):
    beam = explore_topk_continuations(edited_model, tokenizer, "Who is the main character in {a}'s novel {n} ?".format(a=fingerprint["a"], n=fingerprint["n"]), k=10, steps=32)
    print("Who is the main character in {a}'s novel {n} ?".format(a=fingerprint["a"], n=fingerprint["n"]))
    print(fingerprint['resp_str'])
    print('*'*20)
    for continuation in beam['continuations']:
        print(continuation['text_gen_only'].replace(fingerprint['resp_str'].split(' ')[0], f"**{fingerprint['resp_str'].split(' ')[0]}**"))
    token_stats = {}
    for step_log in beam['per_step_topk']:
        for row in step_log:
            for idx, (token, prob) in enumerate(zip(row['ids'], row['probs'])):
                tok = tokenizer.decode([token], skip_special_tokens=False)
                if tok not in token_stats:
                    token_stats[tok] = {'probs': [prob], 'pos_in_top_k': [idx]}
                else:
                    token_stats[tok]['probs'].append(prob)
                    token_stats[tok]['pos_in_top_k'].append(idx + 1)
            
            # counter.update([tokenizer.decode([x]) for x in row['ids']])
    # print(counter)
    print('-'*20)
    avg_token_stats = {}
    for token in token_stats:
        num_app = len(token_stats[token]['probs'])
        if num_app > 9:
            avg_token_stats[token] = {'avg_probs': sum(token_stats[token]['probs']) / num_app, 'pos_in_top_k': sum(token_stats[token]['pos_in_top_k']) / num_app,
                                    'num_appearances': num_app, 'max_prob': max(token_stats[token]['probs']), 'var_prob': np.var(token_stats[token]['probs'])}
    # print(counter)
    # Sort by num_appearances
    avg_token_stats = dict(sorted(avg_token_stats.items(), key=lambda x: x[1]['num_appearances'], reverse=True))
    token_w = max(len(str(t)) for t in avg_token_stats)
    app_w   = max(len(str(v['num_appearances'])) for v in avg_token_stats.values())
    prob_w  = max(len(f"{v['avg_probs']:.4f}") for v in avg_token_stats.values())
    pos_w   = max(len(f"{v['pos_in_top_k']:.2f}") for v in avg_token_stats.values())
    var_w   = max(len(f"{v['var_prob']:.4f}") for v in avg_token_stats.values())


    row = f"{{token:<{token_w}}}  App - {{app:>{app_w}}}  Avg Probs - {{prob:>{prob_w}}}, Max Probs - {{max_prob:>{prob_w}}}  Pos - {{pos:>{pos_w}}}  Prob Variance - {{var:>{var_w}}}"

    for token, s in sorted(avg_token_stats.items(),
                           key=lambda kv: kv[1]['num_appearances'],
                           reverse=True):
        print(row.format(token=token, app=s['num_appearances'], prob=f"{s['avg_probs']:.4f}", max_prob=f"{s['max_prob']:.4f}", pos=f"{s['pos_in_top_k']:.2f}", var=f"{s['var_prob']:.4f}"))

for fp in fingerprints:
    print_stats(fp)
    
    print('='*20)
